# Excel-отчёты через ZEMI Arsenal

Тот же сценарий, что в `runme.ipynb`, но без прямого использования `zemi.exp` и `zemi.toml`. Arsenal загружает и запускает Qwen 3.5 4B, `MarkItDown` переводит Excel в компактный Markdown, а `llama_cpp_agent` ограничивает единственную генерацию GBNF-грамматикой из Pydantic-схемы.

## Подготовка и запуск Arsenal

In [ ]:
from zemi.playbook import Arsenal


arsenal = Arsenal("@comp/playbook.toml")
# arsenal.download()  # Раскомментируйте, чтобы заранее скачать все ресурсы.
arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=False,
)

assistant = (
    arsenal.llamas["primary"]
    .models["qwen"]
    .assistants["report_parser"]
)
provider = assistant.clients.llama_cpp_agent

## Компактная схема результата

`llama_cpp_agent` преобразует Pydantic-схему в GBNF и передаёт грамматику непосредственно llama.cpp. Формат обеспечивается во время одной генерации — повторная отправка полного запроса для исправления результата не используется.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field
from llama_cpp_agent import LlamaCppAgent
from llama_cpp_agent.llm_output_settings.settings import (
    LlmStructuredOutputSettings,
    LlmStructuredOutputType,
)


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Transaction(StrictModel):
    date: str = Field(description="Дата в формате YYYY-MM-DD")
    article: str
    cost: float


class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description="Дата в формате YYYY-MM-DD")
    manager: str
    transactions: list[Transaction]


class Reports(StrictModel):
    reports: list[Report]


structured_output = LlmStructuredOutputSettings.from_pydantic_models(
    [Reports],
    output_type=LlmStructuredOutputType.object_instance,
)
gbnf = structured_output.get_gbnf_grammar()
print(f"GBNF подготовлена: {len(gbnf)} символов")

## Excel → Markdown

In [ ]:
from IPython.display import Markdown, display
from markitdown import MarkItDown

from zemi import env


data_dir = env.path.comp / "data/case01"
excel_files = [data_dir / f"Отчет {number}.xlsx" for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)

excel_context = "\n\n".join(
    f"# Файл: {path.name}\n\n{converter.convert(path).text_content.strip()}"
    for path in excel_files
)

print("Подготовленные файлы:")
for path in excel_files:
    print(f"- {path.name}: {path.stat().st_size} байт")
print(f"\nРазмер Markdown-контекста: {len(excel_context)} символов")
display(Markdown(excel_context))

## Структурированное извлечение

In [ ]:
import json


task = """
Обработай все переданные Excel-отчёты. Для каждого файла извлеки имя файла,
город/филиал, дату выгрузки, руководителя и строки таблицы (date, article, cost).
Не включай строку «Итого» и не выдумывай отсутствующие данные.
""".strip()

client = LlamaCppAgent(
    provider,
    system_prompt=(
        "Ты аккуратно преобразуешь Excel-отчёты в строго "
        "структурированные данные и ничего не выдумываешь."
    ),
)
settings = provider.get_provider_default_settings()
settings.temperature = 0.0
settings.n_predict = 2048
settings.stream = True

result = client.get_chat_response(
    message=f"{task}\n\n{excel_context}",
    structured_output_settings=structured_output,
    llm_sampling_settings=settings,
    add_message_to_chat_history=False,
    add_response_to_chat_history=False,
)

print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))

In [ ]:
result

## Остановка Arsenal

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
arsenal.end_playbook(stop_arsenal_after_end=True)